# Opening datasets


Creating a Combined Comments column

In [ ]:
import pandas as pd

# dataset extracted from the Forever Lobbying Project at https://docs.google.com/spreadsheets/d/1dqcEDUYA9zxx4SLJNfobyfabY0G-OPbgnRbDOHD60QQ/edit?gid=627989861#gid=627989861
df = pd.read_csv('ECHA_COMMENTS_FULL.csv')

# selecting the columns to be combined
columns_to_combine = [
    'General Comments',
    '1. Sectors and (sub-)uses',
    '2. Emissions in the end-of-life phase',
    '3. Emissions in the end-of-life phase',
    '4. Impacts on the recycling industry',
    '5. Proposed derogations',
    '6. Missing uses',
    '7. Potential derogations marked for reconsideration',
    '8. Other identified uses',
    '9. Degradation potential of specific PFAS sub-groups',
    '10. Analytical methods'
]

def combine_comments(row):
    combined = []
    for col in columns_to_combine:
        value = row[col]
        if pd.notna(value) and str(value).strip():
            combined.append(f"{col}: {value.strip()}")
    return "\n\n".join(combined) if combined else None

# Combined Comments column
df['Combined Comments'] = df.apply(combine_comments, axis=1)
df.head()

df


Token count by comment and total

In [ ]:
import tiktoken


In [ ]:
encoding = tiktoken.encoding_for_model("gpt-4o")

df["token_count"] = df["Combined Comments"].apply(
    lambda x: len(encoding.encode(x)) if pd.notna(x) else 0
)

total_tokens = df["token_count"].sum()
print(f"Total tokens: {total_tokens}")


Total tokens: 3711430


## Data Preparation and Exploration

Replacing comments in foreign language by the comments translated by Valeria

In [ ]:
import pandas as pd

echa_df = df
comments_df = pd.read_csv('comments_definitive.csv')

comments_df = comments_df.rename(columns={"Combined Comments": "Combined Comments Translated"})

merged_df = echa_df.merge(
    comments_df[['ID', 'Content', 'Country', 'language', 'Combined Comments Translated']],
    left_on='Ref.', right_on='ID', how='left', suffixes=('', '_new')
)

merged_df['Content'] = merged_df['Content_new']

def get_non_english_comment(row):
    lang = row['language']
    comment = row['Combined Comments Translated']
    if pd.notna(lang) and lang.lower() not in ['en', 'undetermined'] and pd.notna(comment):
        return comment
    return None

merged_df['Combined Comments Non-English'] = merged_df.apply(get_non_english_comment, axis=1)

merged_df['Combined Comments'] = merged_df['Combined Comments Non-English'].combine_first(merged_df['Combined Comments'])

merged_df.drop(columns=['Combined Comments Translated', 'Content_new', 'Ref.'], inplace=True)

cols = ['ID'] + [col for col in merged_df.columns if col != 'ID']
merged_df = merged_df[cols]


#### Exemption request mining

From the "Content" column, where exemption requests are stated

In [ ]:
merged_df["Exemption Request"] = merged_df["Content"].apply(
    lambda x: 1 if isinstance(x, str) and "request for exemption" in x.lower() else 0
)


In [ ]:
merged_df["Exemption Request"].value_counts()

,count
Exemption Request,
1,2991
0,2651


Adding the classified lobbying arguments

In [ ]:
if 'Argument Labels' not in merged_df.columns:
    merged_df['Argument Labels'] = None


category_mapping = {
    "1": ["1.1", "1.2"],
    "2": ["2.1", "2.2"],
    "3": ["3.1", "3.2", "3.3", "3.4"],
    "4": ["4.1", "4.2", "4.3"],
    "5": ["5"],
}

from itertools import chain

def make_full_dummy_df(resp_dict, cat_map):
    all_subs = sorted(set(chain.from_iterable(cat_map.values())))
    rows = []
    for idx, label_str in resp_dict.items():
        subs = [s.strip() for s in label_str.split(",")] if label_str else []
        row = {"ID": idx}
        for top, sub_list in cat_map.items():
            row[f"{top}_argument"] = int(any(s in subs for s in sub_list))
        for sub in all_subs:
            row[f"{sub}_argument"] = int(sub in subs)
        rows.append(row)
    return pd.DataFrame(rows).set_index("ID").astype("uint8")

dummy_df = make_full_dummy_df(responses, category_mapping)

merged_df = merged_df.set_index("ID").join(dummy_df, how="left").reset_index()


In [ ]:
merged_df.columns.tolist()

['ID',
 'Type',
 'Date',
 'Content',
 'Org. country',
 'Org. type',
 'Org. name',
 'Company name confidential',
 'attachment_author',
 'General Comments',
 '1. Sectors and (sub-)uses',
 '2. Emissions in the end-of-life phase',
 '3. Emissions in the end-of-life phase',
 '4. Impacts on the recycling industry',
 '5. Proposed derogations',
 '6. Missing uses',
 '7. Potential derogations marked for reconsideration',
 '8. Other identified uses',
 '9. Degradation potential of specific PFAS sub-groups',
 '10. Analytical methods',
 'source_document',
 'attachments',
 'Combined Comments',
 'token_count',
 'Exemption Request',
 'Country',
 'language',
 'Combined Comments Non-English',
 'Argument Labels',
 '1_argument',
 '2_argument',
 '3_argument',
 '4_argument',
 '5_argument',
 '1.1_argument',
 '1.2_argument',
 '2.1_argument',
 '2.2_argument',
 '3.1_argument',
 '3.2_argument',
 '3.3_argument',
 '3.4_argument',
 '4.1_argument',
 '4.2_argument',
 '4.3_argument',
 'attachment_link_dummy',
 'confiden

Adding sector classification by Zina

In [ ]:
zin_df = pd.read_csv("df_final_zin.csv")

cols_to_add = [
    'Ref.',
    'attachment_link_dummy',
    'confidential_info_dummy',
    'missing_key_info',
    'fine_sector',
    'gpt_stance'
]

zin_filtered = zin_df[cols_to_add].copy()

merged_df = merged_df.merge(
    zin_filtered,
    left_on='ID',
    right_on='Ref.',
    how='left'
)

columns_to_fill = [
    'attachment_link_dummy',
    'confidential_info_dummy',
    'missing_key_info',
    'fine_sector',
    'gpt_stance'
]
merged_df[columns_to_fill] = merged_df[columns_to_fill].fillna("unknown")

merged_df.drop(columns=['Ref.'], inplace=True)


Adding stance classified by Valeria

In [ ]:
valeria_df = pd.read_excel("/content/df_final_pfas_val.xlsx")

valeria_stance = valeria_df[['ID', 'Stance']].rename(columns={'Stance': 'Stance_Valeria'})

merged_df = merged_df.merge(valeria_stance, on='ID', how='left')


In [ ]:
merged_df["fine_sector"].value_counts()

,count
fine_sector,
individual,1340
C20,771
unknown,749
C26,494
C28,390
...,...
C11,1
A1,1
M75,1


In [ ]:
merged_df = merged_df.drop(columns=['Combined Comments Non-English'], errors='ignore')


Mapping NACE labels

In [ ]:
sectors_in_data = set(merged_df["fine_sector"].dropna().unique())
sectors_in_mapping = set(nace_mapping.keys())

missing_codes = sorted(sectors_in_data - sectors_in_mapping)

print("Missing fine_sector codes (not in NACE mapping):")
print(missing_codes)


Missing fine_sector codes (not in NACE mapping):
['individual', 'unknown']


In [ ]:
import pandas as pd

# NACEl abel mapping
nace_df = pd.read_csv("sectors_by_stance_with_description.csv")

# nace code -> label
nace_mapping = nace_df.set_index("fine_sector")["Description"].dropna().to_dict()

nace_mapping.update({
    'individual': 'individual',
    'unknown': 'unknown'
})

merged_df["NACE_label"] = merged_df["fine_sector"].map(nace_mapping)


,count
NACE_label,
individual,1340
Manufacture of chemicals and chemical products,771
unknown,749
"Manufacture of computer, electronic and optical products",494
Manufacture of machinery and equipment n.e.c.,390
...,...
Manufacture of beverages,1
"Crop and animal production, hunting and related service activities",1
Veterinary activities,1


In [ ]:
merged_df["Org. type"] = merged_df["Org. type"].fillna("Individual")
merged_df.loc[merged_df["Org. type"].str.strip() == "", "Org. type"] = "Individual"

,count
Type,
BehalfOfAnOrganisation,4090
Individual,1543
MemberState,9


Nace code mapping to wider categories for the treemap in the dashboard

In [ ]:
nace_section_labels = {
    "A": "Agriculture, forestry and fishing",
    "B": "Mining and quarrying",
    "C": "Manufacturing",
    "D": "Electricity, gas, steam and air conditioning supply",
    "E": "Water supply; sewerage, waste management and remediation activities",
    "F": "Construction",
    "G": "Wholesale and retail trade; repair of motor vehicles and motorcycles",
    "H": "Transportation and storage",
    "I": "Accommodation and food service activities",
    "J": "Information and communication",
    "K": "Financial and insurance activities",
    "L": "Real estate activities",
    "M": "Professional, scientific and technical activities",
    "N": "Administrative and support service activities",
    "O": "Public administration and defence; compulsory social security",
    "P": "Education",
    "Q": "Human health and social work activities",
    "R": "Arts, entertainment and recreation",
    "S": "Other service activities",
    "T": "Activities of households as employers; undifferentiated goods- and services-producing activities of households for own use",
    "U": "Activities of extraterritorial organisations and bodies"
}


In [ ]:
merged_df['NACE_section_code'] = merged_df['fine_sector'].astype(str).str[0]

merged_df['NACE_section_label'] = merged_df['NACE_section_code'].map(nace_section_labels)

merged_df['NACE_section_label'] = merged_df['NACE_section_label'].fillna("Unknown")


,NACE_section_code
0,u
1,C
2,C
3,C
4,C
...,...
5637,u
5638,u
5639,u
5640,J


In [ ]:
merged_df['NACE_section_label'].value_counts()

,count
NACE_section_label,
Manufacturing,2657
Unknown,2089
Information and communication,180
"Professional, scientific and technical activities",179
Administrative and support service activities,159
Other service activities,113
Transportation and storage,87
Activities of households as employers; undifferentiated goods- and services-producing activities of households for own use,51
Mining and quarrying,31


In [ ]:
merged_df.drop(columns=['NACE_section_code'], inplace=True)


,ID,Type,Date,Content,Org. country,Org. type,Org. name,Company name confidential,attachment_author,General Comments,...,4.2_argument,4.3_argument,attachment_link_dummy,confidential_info_dummy,missing_key_info,fine_sector,gpt_stance,Stance_Valeria,NACE_label,NACE_section_label
0,3863,BehalfOfAnOrganisation,2023/03/29 10:55,Transitional period Request for exemption,"Korea, Republic of",Company,<redacted>,Yes,NaN,1,...,0,0,unknown,unknown,unknown,unknown,unknown,NaN,unknown,Unknown
1,6946,BehalfOfAnOrganisation,2023/08/23 11:26,Request for exemption,China,Company,"Huaqin Energy Storage Technology Co., LTD.",NaN,SUOMELA Sanna,\n\n\n\n\nFlow battery membrane is a kind of p...,...,0,0,1.0,0.0,0.0,C,exemption,Against,Manufacturing,Manufacturing
2,7555,BehalfOfAnOrganisation,2023/09/13 14:49,Scope or restriction option analysis Hazard or...,Netherlands,Regional or local authority,Municipality of Dordrecht,NaN,NaN,\n\nBij deze dienen wij een reactie in voor de...,...,0,0,1.0,0.0,0.0,C20,support,In favor,Manufacture of chemicals and chemical products,Manufacturing
3,4565,BehalfOfAnOrganisation,2023/06/15 13:38,Scope or restriction option analysis Hazard or...,France,Company,<redacted>,Yes,NaN,\n\nThe Etienne Lacroix Group is a pyrotechnic...,...,0,1,1.0,0.0,0.0,C30,against,Against,Manufacture of other transport equipment,Manufacturing
4,6818,BehalfOfAnOrganisation,2023/08/21 8:56,Scope or restriction option analysis Informati...,Japan,Industry or trade association,Medical Technology Association of Japan (MTJAPAN),NaN,NaN,"\n We, the Medical Technology Association of J...",...,1,0,0.0,0.0,0.0,C26,exemption,Against,"Manufacture of computer, electronic and optica...",Manufacturing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5637,6547,BehalfOfAnOrganisation,2023/08/07 13:03,Scope or restriction option analysis Hazard or...,Hungary,Company,<redacted>,Yes,NaN,NaN,...,0,0,unknown,unknown,unknown,unknown,unknown,NaN,unknown,Unknown
5638,6581,Individual,2023/08/09 11:52,NaN,NaN,Individual,NaN,NaN,NaN,NaN,...,0,0,unknown,unknown,unknown,unknown,unknown,NaN,unknown,Unknown
5639,7636,Individual,2023/09/15 5:48,Scope or restriction option analysis,NaN,Individual,NaN,NaN,NaN,NaN,...,0,0,unknown,unknown,unknown,unknown,unknown,NaN,unknown,Unknown
5640,9262,BehalfOfAnOrganisation,2023/09/25 13:54,Hazard or exposure Description of analytical m...,Italy,Industry or trade association,Confindustria,NaN,SUOMELA Sanna,NaN,...,0,0,1.0,0.0,0.0,J63,uncertain,Neutral,Information service activities,Information and communication


In [ ]:
merged_df.to_csv("comments_definitive.csv", index=False)

In [ ]:
merged_df["Stance_Valeria"].value_counts()


,count
Stance_Valeria,
Against,2493
In favor,1373
Neutral,905


In [ ]:
merged_df["gpt_stance"].value_counts()

,count
gpt_stance,
exemption,1501
support,1450
against,982
uncertain,960
unknown,749


In [ ]:
unknown_missing_info_df = merged_df[merged_df['missing_key_info'] == 'unknown']

In [ ]:
import pandas as pd

df = pd.read_csv('arguments_LONG.csv')

df["Argument_Label"].value_counts()

,count
Argument_Label,
No arguments,3183
No alternative arguments,2398
No alternative,2365
Economic arguments,2119
Economic impact,1944
Competitiveness loss,1661
Social arguments,1327
Scientific arguments,986
Polymer of Low Concern,970


### Many organisation have multiple/duplicate submissions

In [ ]:
df_named_orgs = df[df['Org. name'] != '<redacted>']

org_counts = df_named_orgs['Org. name'].value_counts()
orgs_with_multiple = org_counts[org_counts > 1].index

org_comment_check = (
    df_named_orgs[df_named_orgs['Org. name'].isin(orgs_with_multiple)]
    .groupby('Org. name')['General Comments']
    .agg(['count', pd.Series.nunique])
    .reset_index()
)

org_comment_check['All Duplicates'] = org_comment_check['nunique'] == 1

org_comment_check.columns = ['Org. name', 'Total Comments', 'Unique Comments', 'All Duplicates']

org_comment_counts_sorted = org_comment_check.sort_values(by='Total Comments', ascending=False)

print(org_comment_counts_sorted)

                               Org. name  Total Comments  Unique Comments  \
197         W. L. Gore & Associates GmbH              27               27   
133                      NTN Corporation              26               26   
1                                    ABB              14               14   
2          ADVANCE ELECTRIC COMPANY INC.              14               12   
41           Daikin Chemical Europe GmbH              14               14   
..                                   ...             ...              ...   
72                      Fothergill Group               1                1   
179                       TEIJIN LIMITED               1                1   
151                     SEMI Europe GmbH               1                1   
23   Berghof Products + Instruments GmbH               0                0   
162        Schreiner Group GmbH & Co. KG               0                0   

     All Duplicates  
197           False  
133           False  
1        

## Swedish submissions

Many swedish commenters have duplicate or copy/paste comments - let's analyse the extent of it by looking at how many of the comments have the first 20 words as the exact same.


In [ ]:
sweden_df = df[df["Country"] == "Sweden"].copy()

sweden_df["first_20_words"] = sweden_df["Combined Comments"].str.split().apply(
    lambda x: " ".join(x[:10]) if isinstance(x, list) else ""
)

duplicate_groups = sweden_df["first_20_words"].value_counts()

repeated_starts = duplicate_groups[duplicate_groups > 1]

matching_comments_count = sweden_df[sweden_df["first_20_words"].isin(repeated_starts.index)].shape[0]

print("Number of comments from Sweden with identical first 20 words:", matching_comments_count)


Number of comments from Sweden with identical first 20 words: 385


With cosine similarity


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sweden_df = df[df["Country"] == "Sweden"].copy()

comments = sweden_df["Combined Comments"].dropna().tolist()

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(comments, convert_to_tensor=True)

similarity_matrix_swed = cosine_similarity(embeddings.cpu().numpy())


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
threshold = 0.9
n = len(comments)
similar_rows_swed = set()

for i in range(n):
    for j in range(i + 1, n):
        if similarity_matrix_swed[i, j] >= threshold:
            similar_rows_swed.add(i)
            similar_rows_swed.add(j)

# number of unique comments with a highly similar counterpart
print("Number of Swedish comments that are semantically very similar:", len(similar_rows_swed))

Number of Swedish comments that are semantically very similar: 503


Same similarity analysis, but for German comments to serve as a comparison

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

germany_df = df[df["Country"] == "Germany"].copy()

comments = germany_df["Combined Comments"].dropna().tolist()

# compute semantic embeddings
embeddings_German = model.encode(comments, convert_to_tensor=True)

# compute cosine similarity matrix
similarity_matrix_Ger = cosine_similarity(embeddings_German.cpu().numpy())


In [ ]:
threshold = 0.9
n = len(comments)
similar_rows_ger = set()

for i in range(n):
    for j in range(i + 1, n):
        if similarity_matrix_Ger[i, j] >= threshold:
            similar_rows_ger.add(i)
            similar_rows_ger.add(j)

# Number of unique comments with a highly similar counterpart
print("Number of German comments that are semantically very similar:", len(similar_rows_ger))

Number of German comments that are semantically very similar: 195


For dashboard

## Long dataframe to be able to create lobbying argument graph in the dashboard

In [ ]:
import pandas as pd

argument_columns = [col for col in merged_df.columns if col.endswith("_argument")]
reshaped_df = merged_df[["ID", "Country", "NACE_label", "Org. type", 'NACE_section_label'] + argument_columns]

long_df = reshaped_df.melt(id_vars=["ID", "Country", "NACE_label", "Org. type", 'NACE_section_label'],
                           value_vars=argument_columns,
                           var_name="Argument_Type",
                           value_name="Flag")

long_df = long_df[long_df["Flag"] == 1].drop(columns=["Flag"])

argument_mapping = {
    "1_argument": "Scientific arguments",
    "1.1_argument": "Polymer of Low Concern",
    "1.2_argument": "Negligible toxicity",
    "2_argument": "No alternative arguments",
    "2.1_argument": "No alternative",
    "2.2_argument": "Alternatives underperform",
    "3_argument": "Economic arguments",
    "3.1_argument": "Economic impact",
    "3.2_argument": "Forced relocation",
    "3.3_argument": "Job loss",
    "3.4_argument": "Competitiveness loss",
    "4_argument": "Social arguments",
    "4.1_argument": "Customer convenience",
    "4.2_argument": "Health and safety",
    "4.3_argument": "Green transition",
    "5_argument": "No arguments"
}
long_df["Argument_Label"] = long_df["Argument_Type"].map(argument_mapping)

In [ ]:
long_df.to_csv("arguments_LONG.csv", index=False)

# Word Cloud for Dashboard: frequency by country

In [ ]:
import pandas as pd
import string
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# sklearn's standard English stopword list
stop_words = ENGLISH_STOP_WORDS

# combine comment fields and clean text
df['Combined Comments'] = df['Combined Comments'].str.lower().str.translate(str.maketrans('', '', string.punctuation))

# compute word frequencies per country
word_freq_data = []
for country, group in df.groupby('Country'):
    full_text = ' '.join(group['Combined Comments'].dropna())
    tokens = full_text.split()
    filtered_tokens = [word for word in tokens if word not in stop_words and word.isalpha()]
    word_counts = Counter(filtered_tokens)
    for word, freq in word_counts.items():
        word_freq_data.append({'Country': country, 'Word': word, 'Frequency': freq})

# create and export result
word_freq_df = pd.DataFrame(word_freq_data)
word_freq_df.to_csv("keyword_frequency_by_country.csv", index=False)
